# Olist Brazilian E-Commerce 데이터 로드
캐글 Olist 데이터셋의 모든 CSV 파일을 불러옵니다.

In [ ]:
import kagglehub
import pandas as pd
import os

# 데이터셋 경로 (캐시된 경로 사용, 없으면 다운로드)
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("데이터 경로:", path)

# 각 CSV 파일 로드
df_customers = pd.read_csv(os.path.join(path, "olist_customers_dataset.csv"))
df_geolocation = pd.read_csv(os.path.join(path, "olist_geolocation_dataset.csv"))
df_orders = pd.read_csv(os.path.join(path, "olist_orders_dataset.csv"))
df_order_items = pd.read_csv(os.path.join(path, "olist_order_items_dataset.csv"))
df_order_payments = pd.read_csv(os.path.join(path, "olist_order_payments_dataset.csv"))
df_order_reviews = pd.read_csv(os.path.join(path, "olist_order_reviews_dataset.csv"))
df_products = pd.read_csv(os.path.join(path, "olist_products_dataset.csv"))
df_sellers = pd.read_csv(os.path.join(path, "olist_sellers_dataset.csv"))
df_category_translation = pd.read_csv(
    os.path.join(path, "product_category_name_translation.csv"),
    encoding="utf-8-sig",  # BOM 제거 (일부 환경에서 컬럼명 앞 ï»¿ 방지)
)

# 한 번에 참조하기 위한 딕셔너리 (선택)
olist = {
    "customers": df_customers,
    "geolocation": df_geolocation,
    "orders": df_orders,
    "order_items": df_order_items,
    "order_payments": df_order_payments,
    "order_reviews": df_order_reviews,
    "products": df_products,
    "sellers": df_sellers,
    "category_translation": df_category_translation,
}

# 로드 결과 요약
for name, df in olist.items():
    print(f"{name}: {df.shape[0]:,}행 × {df.shape[1]}열")

## 1. CSV별 기본 EDA (결측치, 타입, 중복, 요약)
아래 셀 실행 시 각 테이블의 **행/열 수, 결측치, 중복 행, 주요 컬럼 분포**를 한 번에 확인할 수 있습니다.

In [ ]:
# === 기본 EDA: 결측치, 타입, 중복 ===
def eda_summary(df, name):
    print(f"\n{'='*60}\n【 {name} 】 shape: {df.shape[0]:,}행 × {df.shape[1]}열\n{'='*60}")
    print("결측치:")
    nulls = df.isnull().sum()
    print(nulls[nulls > 0].to_string() if nulls.any() else "  없음")
    print("\n중복 행 수:", df.duplicated().sum())
    print("\ndtypes:\n", df.dtypes.to_string())

eda_summary(df_customers, "customers")
eda_summary(df_geolocation, "geolocation")
eda_summary(df_orders, "orders")
eda_summary(df_order_items, "order_items")
eda_summary(df_order_payments, "order_payments")
eda_summary(df_order_reviews, "order_reviews")
eda_summary(df_products, "products")
eda_summary(df_sellers, "sellers")
eda_summary(df_category_translation, "category_translation")

In [ ]:
# === 주요 컬럼 분포 (인사이트용) ===
print("【 orders 】 order_status")
print(df_orders["order_status"].value_counts().to_string())
print("\n【 order_items 】 price/freight 요약")
print(df_order_items[["price", "freight_value"]].describe().round(2))
print("\n【 order_payments 】 payment_type")
print(df_order_payments["payment_type"].value_counts().to_string())
print("\n【 order_reviews 】 review_score")
print(df_order_reviews["review_score"].value_counts().sort_index().to_string())
print("\n【 products 】 컬럼명 확인 (오타 여부)")
print(list(df_products.columns))

## 2. 테이블별 EDA 인사이트 요약

| 테이블 | 인사이트 |
|--------|----------|
| **customers** | 결측·중복 없음. `customer_id`(주문 단위)와 `customer_unique_id`(동일 고객) 구분 필요. |
| **geolocation** | 약 100만 행, **중복 26만 건** — 동일 우편번호에 여러 (lat, lng, city) 조합 존재. 도시명에 **특수문자·숫자·오타**(*, …, 서수표기 등) 포함된 경우 있음. |
| **orders** | **날짜 컬럼이 object** → `pd.to_datetime` 필요. `order_approved_at` 160건, `order_delivered_carrier_date` 1,783건, `order_delivered_customer_date` 2,965건 결측 — 미배송/취소 등 상태와 연관. |
| **order_items** | 주문 수(99,441)보다 행 수(112,650)가 많음 — **1주문 다상품**. price/freight 이상치(0 또는 매우 큰 값) 검토 필요. |
| **order_payments** | 행 수(103,886) > 주문 수 — **1주문 다건 결제**(할부 등). `payment_type`에 `not_defined` 3건. |
| **order_reviews** | 리뷰 행 수(99,224) ≈ 주문 수. **review_comment_title 87,656건, review_comment_message 58,247건 결측** — 텍스트 분석 시 제외/별도 처리. |
| **products** | **컬럼명 오타**: `product_name_lenght` → length, `product_description_lenght` → length. **카테고리·이름길이·설명길이·사진수 610건 동일 결측**, 무게/치수 2건 결측. |
| **sellers** | 결측·중복 없음. |
| **category_translation** | 71개 카테고리. CSV 인코딩에 따라 컬럼명에 BOM(`ï»¿`) 붙을 수 있음 — `encoding='utf-8-sig'` 권장. |

## 3. 전처리 체크리스트 (대부분 진행하는 항목)

- [ ] **orders**: `order_purchase_timestamp`, `order_approved_at`, `order_delivered_*`, `order_estimated_delivery_date` → `pd.to_datetime()` 변환.
- [ ] **orders**: 분석 목적에 따라 `order_status == 'delivered'` 등 **완료/취소 구분** 후 서브셋 사용.
- [ ] **orders**: 배달 소요일 등 파생 변수 생성 시 `order_delivered_customer_date` 결측(2,965건) 처리(제외 또는 별도 플래그).
- [ ] **products**: 컬럼명 수정 — `product_name_lenght` → `product_name_length`, `product_description_lenght` → `product_description_length`.
- [ ] **products**: 카테고리/이름길이/설명길이/사진 수 **610건 동일 결측** — 삭제 vs 'unknown' 등 채우기 결정. 무게/치수 2건 결측 처리.
- [ ] **geolocation**: 동일 `geolocation_zip_code_prefix`당 하나만 쓰려면 **중복 제거**(예: 첫 행만 유지 또는 lat/lng 평균).
- [ ] **geolocation**: 도시명 정제 — **악센트(á, ã, ç 등) 제거/정규화**, 앞뒤 `*`, `…`, 숫자 등 제거, 대소문자 통일(예: 첫 글자만 대문자).
- [ ] **customers / sellers**: 도시명 사용 시 geolocation과 동일한 **정규화 규칙** 적용해 조인 품질 확보.
- [ ] **order_reviews**: 리뷰 텍스트 분석 시 `review_comment_title`, `review_comment_message` 결측 비율 높음 — 결측 제외 또는 별도 분석.
- [ ] **order_payments**: `payment_type == 'not_defined'` 3건 — 제거 또는 'unknown' 처리.
- [ ] **category_translation**: 로드 시 `encoding='utf-8-sig'`로 BOM 제거, 컬럼명 `strip()` 필요 시 적용.
- [ ] **조인**: 주문 기준 마스터 만들 때 `orders` ↔ `order_items` ↔ `products` ↔ `sellers` ↔ `order_payments` ↔ `order_reviews` ↔ `customers` **키 정리**(order_id, customer_id, product_id, seller_id). Left/Inner 조인 목적에 따라 선택.

## 4. 주의사항 & 참고사항

### 주의사항
1. **날짜는 전부 object**  
   비교·기간 계산 전에 반드시 `pd.to_datetime()` 변환. 시간대는 브라질(UTC-3 등) 기준일 수 있음.
2. **1:N 관계**  
   - 한 주문에 여러 상품(`order_items`), 여러 결제(`order_payments`). 집계 시 중복 카운트 주의.
   - 한 고객이 여러 주문 → `customer_unique_id`로 고객 단위 분석.
3. **결측의 의미**  
   - `order_delivered_customer_date` 결측: 미배송·취소·진행 중 등. 분석 시 제외할지, 별도 그룹으로 둘지 결정.
   - 리뷰 코멘트 결측: 작성 안 한 경우. 점수만 쓰면 무시 가능.
4. **products 컬럼명 오타**  
   `product_name_lenght`, `product_description_lenght`는 원본 그대로이므로, 전처리 단계에서 rename 권장.
5. **geolocation 중복**  
   우편번호 하나에 여러 (lat, lng, city). 거리/지도 분석 시 하나로 줄이거나, 용도에 맞는 기준 선택.

### 참고사항
- **기간**: 2016~2018년 브라질 이커머스 주문 약 10만 건.
- **언어**: 상품 카테고리·리뷰 등은 **포르투갈어**. `product_category_name_translation.csv`로 영어 매핑 가능.
- **개인정보**: 고객/판매자 ID는 익명화. Game of Thrones 캐릭터 이름 등으로 대체된 부분 있음.
- **리뷰 점수**: 1~5. 만족도·NPS 분석에 자주 사용됨.